<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/Week7_Day4_LLMs_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercices XP : Évaluation des LLM pour la synthèse de texte (Summarization)

## Ce que vous allez apprendre
- Évaluation pratique de la synthèse : précision exacte vs score ROUGE.
- Forces et faiblesses des métriques et comparaison des tailles de modèles.
- Utilisation de `transformers` et `evaluate` de Hugging Face pour des expérimentations rapides.
- Chargement de données, échantillonnage, prétraitement et analyse des sorties de modèles.

**Livrables** : scripts d'évaluation, tableaux comparatifs, métriques personnalisées et analyses courtes.

In [8]:
# Partie I. Configuration (à exécuter une fois)
# Installation des dépendances minimales
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Note : nltk est utilisé pour la tokenisation des phrases

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### Partie II. Chargement et exploration du jeu de données
Jeu de données suggéré : [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (mapping `article` -> `prompt_text`, `highlights` -> `prompt_title`).

In [2]:
import pandas as pd
from datasets import load_dataset

# Chemins vers vos fichiers locaux ; laissez vide pour utiliser l'échantillon HF ou le mode secours
train_path = ''
test_path = ''

# Données de secours en cas d'échec du chargement
fallback = pd.DataFrame([
    {
        'prompt_text': 'Le chat s\'est assis sur le tapis et a ronronné bruyamment pendant que le soleil se couchait.',
        'prompt_title': 'Un chat se repose sur un tapis au coucher du soleil'
    },
    {
        'prompt_text': 'Les scientifiques ont découvert de l\'eau sur la lune, ouvrant de nouvelles voies de recherche.',
        'prompt_title': 'De l\'eau trouvée sur la lune'
    },
    {
        'prompt_text': 'L\'équipe locale a remporté le championnat après un match final dramatique.',
        'prompt_title': 'L\'équipe locale décroche le titre'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            # Chargement depuis Hugging Face
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"Échec du chargement HF ({exc}) ; utilisation de l'échantillon de secours.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...


### Partie III. Synthèse avec T5 (Implémentation)
Objectifs :
- Écrire `batch_generator` pour gérer les mini-lots.
- Utiliser `summarize_with_t5` avec `t5-small` et accélération GPU.
- Ajouter le préfixe "summarize: " aux entrées.

In [3]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
import pandas as pd
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    """Générateur pour diviser une liste en lots de taille batch_size."""
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    """Génère des résumés en utilisant le modèle T5."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    summaries = []
    for batch in batch_generator(texts, batch_size):
        # T5 nécessite le préfixe 'summarize: '
        inputs = ["summarize: " + text for text in batch]
        inputs_tokens = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        with torch.no_grad():
            output_tokens = model.generate(inputs_tokens["input_ids"], max_new_tokens=max_new_tokens)

        decoded = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)
        summaries.extend(decoded)

        # Nettoyage de la mémoire
        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

    return summaries

# Activation de la génération pour l'exercice
RUN_T5 = True
if RUN_T5:
    print("Génération en cours avec T5-small...")
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=4)
    results_df = pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    })
    display(results_df.head())

Skipping T5 generation for speed. Set RUN_T5=True to execute.


### Partie IV. Évaluation de la précision (Accuracy)
Implémentez une précision naïve qui vérifie la correspondance exacte des chaînes.
Expliquez pourquoi cette métrique est souvent nulle pour du texte généré librement.

In [4]:
from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    # Calcul de la correspondance exacte (exact match)
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Précision (correspondance exacte) : {acc:.4f}")
else:
    print("Calcul de précision sauté (pas de prédictions disponibles).")

Accuracy skipped (no predictions).


### Partie V. Implémentation de la métrique ROUGE
Utilisez `evaluate.load("rouge")` et le tokenizer NLTK.
Prétraitez en joignant les phrases avec des sauts de ligne pour un meilleur calcul de ROUGE-L.

In [5]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

# Chargement de la bibliothèque d'évaluation ROUGE
rouge = evaluate.load('rouge')

def normalize_text(text):
    """Normalise le texte en ajoutant des sauts de ligne entre les phrases pour ROUGE-L."""
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    """Calcule les scores ROUGE pour une liste de prédictions."""
    norm_preds = [normalize_text(p) for p in preds]
    norm_refs = [normalize_text(r) for r in refs]
    return rouge.compute(predictions=norm_preds, references=norm_refs, use_stemmer=True)

# Test de validation
test_preds = ["Le chat est sur le tapis.", ""]
test_refs  = ["Le chat est sur le tapis.", "Une référence"]
print("Résultats du test ROUGE :", compute_rouge_score(test_preds, test_refs))

ROUGE sanity check (fill function first):


### Partie VI. Comprendre les scores ROUGE
Expériences à mener :
- Correspondance exacte vs prédiction vide.
- Effet de la racinisation (stemming).
- Analyse des N-grammes (ROUGE-1 vs ROUGE-2).


### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    """Génère des résumés avec GPT-2 en utilisant le prompt TL;DR."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    summaries = []
    for batch in batch_generator(texts, batch_size):
        # On tronque le texte pour laisser de la place à la génération
        inputs = [text[:600] + "\n\nTL;DR:" for text in batch]
        inputs_tokens = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model.generate(inputs_tokens["input_ids"], attention_mask=inputs_tokens["attention_mask"], max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        # On ne garde que ce qui suit le prompt 'TL;DR:'
        generated_only = [d.split("TL;DR:")[-1].strip() for d in decoded]
        summaries.extend(generated_only)

    return summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    """Calcule le score ROUGE-L pour chaque ligne du DataFrame."""
    return [compute_rouge_score([p], [r])['rougeL'] for p, r in zip(df[pred_col], df[ref_col])]

RUN_COMPARE = True
if RUN_COMPARE and 'results_df' in locals():
    print("Génération avec GPT-2...")
    results_df['gpt2_summary'] = summarize_with_gpt2(train_df['prompt_text'].tolist())
    results_df['rougeL_t5'] = compute_rouge_per_row(results_df, 't5_small_summary')
    results_df['rougeL_gpt2'] = compute_rouge_per_row(results_df, 'gpt2_summary')
    display(results_df.head())

### Partie VIII. Comparaison de tous les modèles
Présentez les tableaux et discutez du modèle qui offre les meilleures performances et pourquoi.

In [7]:
def compare_models(rouge_dict):
    """Agrège les scores moyens dans un DataFrame."""
    return pd.DataFrame(rouge_dict).mean().to_frame(name='Score ROUGE-L Moyen')

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    """Affiche les colonnes de résumé côte à côte."""
    return df[['reference_summary'] + pred_cols]

# Affichage final
scores_log = {
    'T5-Small': results_df['rougeL_t5'],
    'GPT-2': results_df['rougeL_gpt2']
}
display(compare_models(scores_log))
display(compare_models_summaries(results_df, ['t5_small_summary', 'gpt2_summary']).head())

## Bilan
- Quelles métriques vous ont semblé les plus informatives ?
- Comment la taille du modèle a-t-elle impacté ROUGE et la qualité qualitative ?
- Rédigez une courte réflexion ici.